In [3]:
!apt-get remove chromium-chromedriver chromium-browser -y
!apt-get autoremove -y

!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!rm google-chrome-stable_current_amd64.deb

!pip install selenium pandas

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libat

In [4]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')

driver = webdriver.Chrome(options=chrome_options)
print("WebDriver initialized successfully!")

WebDriver initialized successfully!


In [5]:
base_url = "https://ikman.lk/en/ads/sri-lanka/cars?enum.condition=used"
ad_urls = []

# Collect Ad URLs from the first 120 pages
for page in range(1, 121):
    print(f"Gathering vehicle links from page {page}...")
    driver.get(f"{base_url}&page={page}")

    links = driver.find_elements(By.TAG_NAME, "a")
    for link in links:
        href = link.get_attribute('href')
        if href and '/en/ad/' in href and href not in ad_urls:
            ad_urls.append(href)

print(f"Finished! Found {len(ad_urls)} unique vehicle ads.")

Gathering vehicle links from page 1...
Gathering vehicle links from page 2...
Gathering vehicle links from page 3...
Gathering vehicle links from page 4...
Gathering vehicle links from page 5...
Gathering vehicle links from page 6...
Gathering vehicle links from page 7...
Gathering vehicle links from page 8...
Gathering vehicle links from page 9...
Gathering vehicle links from page 10...
Gathering vehicle links from page 11...
Gathering vehicle links from page 12...
Gathering vehicle links from page 13...
Gathering vehicle links from page 14...
Gathering vehicle links from page 15...
Gathering vehicle links from page 16...
Gathering vehicle links from page 17...
Gathering vehicle links from page 18...
Gathering vehicle links from page 19...
Gathering vehicle links from page 20...
Gathering vehicle links from page 21...
Gathering vehicle links from page 22...
Gathering vehicle links from page 23...
Gathering vehicle links from page 24...
Gathering vehicle links from page 25...
Gathering

In [6]:
from tqdm.notebook import tqdm

car_data = []

print(f"Starting JSON-direct extraction for {len(ad_urls)} ads...")

try:
    for url in tqdm(ad_urls, desc="Scraping Ads"):
        driver.get(url)

        car_info = {}

        try:
            initial_data = driver.execute_script("return window.initialData;")
            ad_data = initial_data.get('adDetail', {}).get('data', {}).get('ad', {})

            money_data = ad_data.get('money', {})
            raw_price = money_data.get('amount', '')
            if raw_price:
                car_info["Selling Price"] = raw_price.replace("Rs", "").replace(",", "").strip()
            else:
                car_info["Selling Price"] = None

            properties_list = ad_data.get('properties', [])
            props_dict = {prop.get('label'): prop.get('value') for prop in properties_list}

            target_features = [
                "Brand", "Model", "Trim / Edition", "Year of Manufacture",
                "Condition", "Transmission", "Body type", "Fuel type",
                "Engine capacity", "Mileage"
            ]

            for feature in target_features:
                car_info[feature] = props_dict.get(feature)

        except Exception as e:
            print(f"Error extracting JSON from {url}: {e}")

        car_data.append(car_info)

finally:
    driver.quit()
    print("\nWebDriver closed. JSON extraction finished.")

Starting JSON-direct extraction for 2711 ads...


Scraping Ads:   0%|          | 0/2711 [00:00<?, ?it/s]


WebDriver closed. JSON extraction finished.


In [7]:
df = pd.DataFrame(car_data)

if "Mileage" in df.columns:
    df["Mileage"] = df["Mileage"].str.replace(" km", "").str.replace(",", "").astype(float)

if "Engine capacity" in df.columns:
    df["Engine capacity"] = df["Engine capacity"].str.replace(" cc", "").str.replace(",", "").astype(float)

if "Selling Price" in df.columns:
    df["Selling Price"] = pd.to_numeric(df["Selling Price"], errors='coerce')

# Drop rows missing the target variable
df = df.dropna(subset=['Selling Price'])

df.to_csv("raw_used_cars.csv", index=False)
print("\nData successfully cleaned and saved to 'raw_used_cars.csv'!")

df.head()


Data successfully cleaned and saved to 'raw_used_cars.csv'!


,Selling Price,Brand,Model,Trim / Edition,Year of Manufacture,Condition,Transmission,Body type,Fuel type,Engine capacity,Mileage
0,4900000.0,DFSK,Glory,330,2018,Used,Manual,MPV,Petrol,1500.0,90000.0
1,29500000.0,Land Rover,Range Rover Velar,R Dynamic,2018,Used,Automatic,SUV / 4x4,Diesel,1990.0,127000.0
2,47000000.0,Mercedes Benz,S500,S500 PLUS HYBRID,2015,Used,Tiptronic,Saloon,Petrol,2996.0,112000.0
3,4000000.0,Suzuki,A-Star,None,2012,Used,Manual,Hatchback,Petrol,1000.0,120000.0
4,9450000.0,Honda,Grace,EX,2017,Used,Automatic,None,Hybrid,1500.0,142098.0
